# Phase 9: Wave-Level Generation (WLG)
## The Universal Generation Engine — Thinking in Waves, Not Bytes

**Phase 8** proved FLUX can generate bytes. **Phase 9** replaces byte-by-byte generation with wave-by-wave generation — the same mechanism that will later drive image, audio, molecular, and sensor generation.

```
Phase 8:  Field → GRU → b → y → t → e → s  (serial, text-only, ~100 steps)
Phase 9:  Field → wave → wave → wave → wave  (parallel-decodable, universal, ~15 steps)
                    ↓       ↓       ↓       ↓
                  "The"  "future"  "of"   "AI"   (last-mile WaveToText)
```

**New components:** WaveChunker, WaveGenerator (universal core), WaveToText (text last-mile), ThermodynamicWaveSampler

**Runtime:** ~4-6 hours total (Stage 1: ~15min, Stage 2: ~4-6hr)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1 — Clone / Pull FLUX Repository
# ═══════════════════════════════════════════════════════════════
import os

REPO_URL = "https://github.com/Unseengap/FLUX.git"
REPO_DIR = "/content/FLUX"

if os.path.exists(REPO_DIR):
    print("  ℹ Repository exists — pulling latest...")
    os.chdir(REPO_DIR)
    !git pull --ff-only || echo "  ⚠ Pull failed — using existing code"
else:
    print("  ℹ Cloning repository...")
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f"  ✓ Working directory: {os.getcwd()}")
!ls phases/phase9/

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2 — Install Dependencies + Setup
# ═══════════════════════════════════════════════════════════════
!pip install -q torch numpy scipy matplotlib tqdm rich transformers huggingface_hub datasets

# Install FLUX package in editable mode
!pip install -q -e .

print("  ✓ Dependencies installed")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3 — Init PhaseLogger + Hardware + Secrets
# ═══════════════════════════════════════════════════════════════
import sys
import torch
from pathlib import Path

# Ensure paths are set
PROJECT_ROOT = Path("/content/FLUX")
sys.path.insert(0, str(PROJECT_ROOT))
for phase in ['phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase6', 'phase7', 'phase8', 'phase9']:
    sys.path.insert(0, str(PROJECT_ROOT / 'phases' / phase))

from flux_utils import get_device, PhaseLogger, PhaseResults, _resolve_hf_token

log = PhaseLogger(phase=9)
log.separator("Phase 9: Wave-Level Generation")
log.cell_start("Cell 3 — Hardware & Secrets")

# Hardware
device = get_device()
log.info(f"Device: {device}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    log.info(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    log.warning("No GPU detected — training will be slow")

# HuggingFace token
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    log.success("HF_TOKEN loaded from Colab secrets")
except Exception:
    hf_token = _resolve_hf_token()
    if hf_token:
        log.success("HF_TOKEN resolved")
    else:
        log.warning("No HF_TOKEN — uploads will be skipped")

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4 — Load FLUX Components (Phase 7 → Phase 8 fallback → Fresh)
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 4 — Load FLUX Components")

from flux_large import FLUXLarge
from flux_utils import checkpoint_exists, load_checkpoint

# Phase 9 is a PEER to Phase 8, not a continuation.
# Both build on Phase 7 (CSE, field, GR, TL, CGN, memory, bridges).
# Phase 8 added WaveDecoder (byte-level). Phase 9 adds WaveGenerator (wave-level).
# Phase 7 is the true dependency. Phase 8 is only a legacy fallback.

model = None

# 1. Try Phase 7 — the true foundation (has all Phases 1-7 trained)
if model is None:
    try:
        model = FLUXLarge.from_phase7_checkpoint(device=device)
        print("  ✓ Loaded Phase 7 (CSE, field, GR, TL, CGN, memory, bridges)")
        print("  ℹ Phase 9 builds wave-level generation on Phase 7 foundation")
    except Exception as e:
        print(f"  ℹ No Phase 7 checkpoint: {e}")

# 2. Legacy fallback — Phase 8 contains Phase 7 weights + WaveDecoder we don't use
if model is None:
    try:
        model = FLUXLarge.from_phase8_checkpoint(device=device)
        print("  ✓ Loaded Phase 8 as fallback (contains Phase 7 weights)")
        print("  ✗ WaveDecoder ignored — Phase 9 replaces it")
    except Exception as e:
        print(f"  ℹ No Phase 8 checkpoint: {e}")

# 3. Last resort — fresh init (untrained, poor quality)
if model is None:
    print("  ⚠ No checkpoints — using fresh FLUXLarge (untrained)")
    model = FLUXLarge(device=device)

# Freeze everything — Phase 9 only trains WaveGenerator + WaveToText
for param in model.parameters():
    param.requires_grad = False

# Quick smoke test
with torch.no_grad():
    wave = model.cse.encode("Hello world")
    wave_seq = wave.full.to(device)
    wave_vec = wave_seq.mean(dim=0)
    field_feats, sims, locs = model.field.query(wave_vec, k=4)
    merged = field_feats.mean(dim=0) + model.cgn(field_feats.mean(dim=0))

print(f"  ✓ Smoke test: CSE→{wave_seq.shape}, Field→{field_feats.shape}, Merged→{merged.shape}")

log.cell_end("Cell 4 — Load FLUX Components", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5 — Build Phase 9 Modules + Load Training Data
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 5 — Build Phase 9 Modules")

from train_wave_gen import (
    build_phase9_modules, Phase9Trainer, load_training_data,
    generate_text, PHASE9_CONFIG,
)
from wave_sampler import ThermodynamicWaveSampler

# Build fresh Phase 9 modules
chunker, generator, wtt = build_phase9_modules(device=device)

# Load training data — FLUX uses continuous stream learning (no epochs).
# We need enough documents for a single pass to reach max_steps.
# ~10 chunks/doc, batch_size=32 → need ~max_steps * 32/10 docs.
# WTT converges fast (~10K steps) since CSE waves already encode words.
texts = load_training_data(max_docs=250_000)
split_idx = int(len(texts) * 0.9)
train_texts = texts[:split_idx]
eval_texts = texts[split_idx:]
print(f"  Train: {len(train_texts):,} docs, Eval: {len(eval_texts):,} docs")
chunks_est = len(train_texts) * 10
steps_est = chunks_est // 32
print(f"  Estimated chunks: ~{chunks_est:,} → ~{steps_est:,} steps (single pass)")

# Create trainer
trainer = Phase9Trainer(
    flux_model=model,
    wave_chunker=chunker,
    wave_generator=generator,
    wave_to_text=wtt,
    lr=3e-4,
    grad_accum=4,
    log=log,
)

log.cell_end("Cell 5 — Build Phase 9 Modules", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6 — Stage 1: WaveToText Pre-Training (~2-3 hours)
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 6 — Stage 1: WaveToText Pre-Training")

wtt_result = trainer.train_wave_to_text(
    train_texts,
    max_steps=10000,
    batch_size=32,
    log_interval=2000,
)

log.metric("wtt_steps", str(wtt_result.total_steps))
log.metric("wtt_word_accuracy", f"{wtt_result.word_accuracy:.1%}")
log.metric("wtt_final_loss", f"{wtt_result.final_loss:.4f}")
log.metric("wtt_time", f"{wtt_result.total_time_seconds:.1f}s")

log.cell_end("Cell 6 — Stage 1: WaveToText Pre-Training", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6b — DEBUG: WaveToText Decode Diagnostic
# ═══════════════════════════════════════════════════════════════
import torch

wtt.eval()
chunker.eval()

# Pick a few short test words
test_texts = [
    "Hello world",
    "The cat sat on the mat",
    "artificial intelligence",
    "energy",
    "physics",
]

print("═" * 70)
print("  WaveToText Decode Diagnostic")
print("═" * 70)

for text in test_texts:
    with torch.no_grad():
        wave = model.cse.encode(text)
        wave_seq = wave.full.to(device)
        text_bytes = text.encode('utf-8', errors='replace')

        pairs = chunker.chunk_with_bytes(wave_seq, text_bytes)

    print(f"\n  Text: \"{text}\"")
    print(f"  Chunks: {len(pairs)}")

    for i, (chunk_wave, chunk_bytes) in enumerate(pairs[:5]):
        expected = chunk_bytes.decode('utf-8', errors='replace')

        # Test 1: Teacher-forced logits — does the model "know" the answer?
        with torch.no_grad():
            byte_tensor = torch.tensor(list(chunk_bytes), dtype=torch.long, device=device)
            logits = wtt(chunk_wave, byte_tensor)  # [chunk_len, 257]
            preds = logits.argmax(dim=-1)
            target_bytes = list(chunk_bytes)
            correct_bytes = sum(1 for p, t in zip(preds.tolist(), target_bytes) if p == t)
            tf_acc = correct_bytes / len(target_bytes)

        # Test 2: Greedy decode (temperature=0.01 ≈ argmax)
        with torch.no_grad():
            decoded_greedy = wtt.decode(chunk_wave, temperature=0.01)
            greedy_text = decoded_greedy.decode('utf-8', errors='replace')

        # Test 3: Normal decode (temperature=0.5 — what eval uses)
        with torch.no_grad():
            decoded_t05 = wtt.decode(chunk_wave, temperature=0.5)
            t05_text = decoded_t05.decode('utf-8', errors='replace')

        # Test 4: Check raw logits at first step
        with torch.no_grad():
            hidden = wtt.wave_to_hidden(chunk_wave).unsqueeze(0).unsqueeze(0)
            bos_token = torch.full((1,), wtt.BOS, dtype=torch.long, device=device)
            embedded = wtt.byte_embed(bos_token).unsqueeze(0)
            output, _ = wtt.gru(embedded, hidden)
            first_logits = wtt.output_proj(output.squeeze(0).squeeze(0))
            top5 = torch.topk(first_logits, 5)
            top5_bytes = [(idx.item(), f"'{chr(idx.item())}'" if idx.item() < 128 else f"0x{idx.item():02x}")
                          for idx in top5.indices]
            expected_first = chunk_bytes[0] if chunk_bytes else -1

        exact_match = "✓" if decoded_greedy == chunk_bytes else "✗"

        print(f"\n    Chunk {i}: expected=\"{expected}\" ({len(chunk_bytes)} bytes)")
        print(f"      Teacher-forced byte acc: {tf_acc:.0%} ({correct_bytes}/{len(target_bytes)})")
        print(f"      Greedy decode:   \"{greedy_text}\" {exact_match}")
        print(f"      Temp=0.5 decode: \"{t05_text}\"")
        print(f"      First byte — expected: {expected_first} ('{chr(expected_first)}')")
        print(f"      First byte — top5 logits: {top5_bytes}")
        print(f"      First byte — top5 probs:  {torch.softmax(first_logits, -1)[top5.indices].tolist()}")

wtt.train()
chunker.train()

print("\n" + "═" * 70)
print("  If teacher-forced acc is high but greedy fails → exposure bias")
print("  If teacher-forced acc is low → training didn't work (check loss)")
print("  If greedy works but temp=0.5 fails → temperature too high")
print("  If first-byte top5 is wrong → wave→hidden projection is broken")
print("═" * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 7 — Stage 2: WaveGenerator Training (~4-6 hours)
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 7 — Stage 2: WaveGenerator Training")

wg_result = trainer.train_wave_generator(
    train_texts,
    max_steps=5000,
    log_interval=50,
)

log.metric("wg_steps", str(wg_result.total_steps))
log.metric("wg_final_loss", f"{wg_result.final_loss:.4f}")
log.metric("wg_cosine_accuracy", f"{wg_result.wave_cosine_accuracy:.3f}")
log.metric("wg_time", f"{wg_result.total_time_seconds:.1f}s")

total_time = wtt_result.total_time_seconds + wg_result.total_time_seconds
print(f"\n  Total training time: {total_time:.1f}s ({total_time/3600:.1f}h)")

log.cell_end("Cell 7 — Stage 2: WaveGenerator Training", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 8 — Save Checkpoint + Upload to HuggingFace Hub
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 8 — Save & Upload Checkpoint")

from flux_utils import upload_checkpoint_to_hf

# Quick valid word rate evaluation for checkpoint
prompts = [
    "The future of artificial intelligence",
    "In the beginning",
    "Scientists have discovered",
    "The relationship between energy and matter",
    "Modern technology relies on",
]
valid_words = 0
total_words = 0
for p in prompts:
    try:
        result = generate_text(p, model, chunker, generator, wtt, max_waves=15, temperature=0.8)
        continuation = result[len(p):].strip()
        for w in continuation.split():
            clean = w.strip('.,;:!?"\'-()[]{}').lower()
            if clean.isalpha() and len(clean) >= 2:
                total_words += 1
                if len(clean) <= 15:
                    valid_words += 1
    except Exception:
        pass
valid_rate = valid_words / max(total_words, 1)
print(f"  Valid word rate: {valid_rate:.1%}")

# Save checkpoint
ckpt_path = trainer.save_phase9_checkpoint(wtt_result, wg_result, valid_rate)

# Upload to HuggingFace Hub
if hf_token:
    try:
        upload_checkpoint_to_hf(phase=9, hf_token=hf_token)
        log.success("Checkpoint uploaded to HuggingFace Hub")
    except Exception as e:
        log.warning(f"HF upload failed: {e}")
else:
    log.warning("No HF token — skipping upload")

log.cell_end("Cell 8 — Save & Upload Checkpoint", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 9 — Test 1: Wave Distribution Match
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 9 — Test 1: Wave Distribution Match")

import test_phase9_test1
test1_passed = test_phase9_test1.main()
status1 = "PASS" if test1_passed else "FAIL"

log.cell_end("Cell 9 — Test 1: Wave Distribution Match", status1)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 10 — Test 2: Word Reconstruction Accuracy
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 10 — Test 2: Word Reconstruction")

import test_phase9_test2
test2_passed = test_phase9_test2.main()
status2 = "PASS" if test2_passed else "FAIL"

log.cell_end("Cell 10 — Test 2: Word Reconstruction", status2)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 11 — Test 3: Full Pipeline English Words
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 11 — Test 3: Full Pipeline English Words")

import test_phase9_test3
test3_passed = test_phase9_test3.main()
status3 = "PASS" if test3_passed else "FAIL"

log.cell_end("Cell 11 — Test 3: Full Pipeline English Words", status3)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 12 — Demo 1: Wave-Level Generation
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 12 — Demo 1: Wave-Level Generation")

import demo_phase9_demo1
demo_phase9_demo1.main()

log.cell_end("Cell 12 — Demo 1: Wave-Level Generation", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 13 — Demo 2: Wave Sequence Visualization
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 13 — Demo 2: Wave Sequence Visualization")

import demo_phase9_demo2
demo_phase9_demo2.main()

# Display the saved figure in notebook
from IPython.display import Image, display
fig_path = Path("phases/phase9/wave_sequence.png")
if fig_path.exists():
    display(Image(filename=str(fig_path), width=900))

log.cell_end("Cell 13 — Demo 2: Wave Sequence Visualization", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 14 — Demo 3: Thermodynamic Sampling Trace
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 14 — Demo 3: Thermodynamic Sampling Trace")

import demo_phase9_demo3
demo_phase9_demo3.main()

# Display the saved figure in notebook
fig_path = Path("phases/phase9/thermo_wave_sampling.png")
if fig_path.exists():
    display(Image(filename=str(fig_path), width=900))

log.cell_end("Cell 14 — Demo 3: Thermodynamic Sampling Trace", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 15 — Interactive Exploration
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 15 — Interactive Exploration")

# Try your own prompts!
test_prompts = [
    "Hello, my name is",
    "The meaning of life is",
    "In a galaxy far far away",
    "Artificial intelligence will",
    "The ocean is vast and",
]

print("═" * 60)
print("  Interactive Wave Generation")
print("═" * 60)

for prompt in test_prompts:
    try:
        result = generate_text(
            prompt, model, chunker, generator, wtt,
            max_waves=15, temperature=0.8, use_sampler=True,
        )
        print(f"\n  Prompt: \"{prompt}\"")
        print(f"  Output: {result[:200]}")
    except Exception as e:
        print(f"\n  Prompt: \"{prompt}\"")
        print(f"  Error: {e}")

log.cell_end("Cell 15 — Interactive Exploration", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 16 — Results Summary
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 16 — Results Summary")

results = PhaseResults(phase=9, component_name="Wave-Level Generation")

# Training metrics
results.add_metric("WTT training steps", wtt_result.total_steps)
results.add_metric("WTT word accuracy", f"{wtt_result.word_accuracy:.1%}")
results.add_metric("WTT final loss", f"{wtt_result.final_loss:.4f}")
results.add_metric("WG training steps", wg_result.total_steps)
results.add_metric("WG final loss", f"{wg_result.final_loss:.4f}")
results.add_metric("WG cosine accuracy", f"{wg_result.wave_cosine_accuracy:.3f}")
results.add_metric("Valid word rate", f"{valid_rate:.1%}")
results.add_metric("Total training time", f"{total_time:.1f}s ({total_time/3600:.1f}h)")

# Test results
results.add_test("Wave Distribution Match", passed=test1_passed, score=0.0, threshold=0.5)
results.add_test("Word Reconstruction", passed=test2_passed, score=0.0, threshold=0.8)
results.add_test("Full Pipeline English", passed=test3_passed, score=0.0, threshold=0.5)

# Demo results
results.add_demo("Wave-Level Generation", ran=True, quality="See output above")
results.add_demo("Wave Sequence Visualization", ran=True, quality="See figure above")
results.add_demo("Thermodynamic Sampling Trace", ran=True, quality="See figure above")

md_output = results.save()
print(md_output)

log.cell_end("Cell 16 — Results Summary", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 17 — View Full Log
# ═══════════════════════════════════════════════════════════════
print(log.get_log_contents())

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 18 — Final Upload (Logs → HF + GitHub)
# ═══════════════════════════════════════════════════════════════
log.cell_start("Cell 18 — Final Upload")

from flux_utils import upload_logs_to_hf, git_commit_and_push

# Upload logs to HuggingFace
if hf_token:
    try:
        upload_logs_to_hf(phase=9, hf_token=hf_token)
        log.success("Logs uploaded to HuggingFace Hub")
    except Exception as e:
        log.warning(f"Log upload failed: {e}")

# Git commit and push
try:
    git_commit_and_push(
        files=[
            'logs/phase9.log',
            'phases/phase9/RESULTS_PHASE_9.md',
            'results/RESULTS_PHASE_9.md',
        ],
        message='Phase 9: Wave-Level Generation complete',
    )
    log.success("Committed and pushed to GitHub")
except Exception as e:
    log.warning(f"Git push failed: {e}")

log.cell_end("Cell 18 — Final Upload", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 19 — Save Artifacts to Drive
# ═══════════════════════════════════════════════════════════════
import shutil

# Copy key artifacts to Colab output (or Drive)
output_dir = Path("/content/phase9_output")
output_dir.mkdir(exist_ok=True)

# Checkpoint
ckpt_src = Path("checkpoints/phase9.phase.pt")
if ckpt_src.exists():
    shutil.copy2(ckpt_src, output_dir / "phase9.phase.pt")
    print(f"  ✓ Checkpoint copied to {output_dir}")

# Results
for results_file in ["phases/phase9/RESULTS_PHASE_9.md", "results/RESULTS_PHASE_9.md"]:
    src = Path(results_file)
    if src.exists():
        shutil.copy2(src, output_dir / src.name)

# Log
log_src = Path("logs/phase9.log")
if log_src.exists():
    shutil.copy2(log_src, output_dir / "phase9.log")

# Figures
for fig in ["phases/phase9/wave_sequence.png", "phases/phase9/thermo_wave_sampling.png"]:
    src = Path(fig)
    if src.exists():
        shutil.copy2(src, output_dir / src.name)

print(f"  ✓ All artifacts saved to {output_dir}")
print(f"  Contents: {list(output_dir.iterdir())}")

# Try mounting Google Drive and copying there too
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = Path("/content/drive/MyDrive/FLUX_checkpoints")
    drive_dir.mkdir(parents=True, exist_ok=True)
    if ckpt_src.exists():
        shutil.copy2(ckpt_src, drive_dir / "phase9.phase.pt")
        print(f"  ✓ Checkpoint also saved to Google Drive: {drive_dir}")
except Exception as e:
    print(f"  ℹ Drive not mounted: {e}")

print("\n" + "═" * 60)
print("  Phase 9: Wave-Level Generation — COMPLETE")
print("═" * 60)